# 模型加载、部署、量化、剪枝、微调

### 检查所需的库

In [1]:
pip show torch transformers datasets peft accelerate

Name: torch
Version: 2.3.1+cu118
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org/
Author: PyTorch Team
Author-email: packages@pytorch.org
License: BSD-3
Location: d:\anaconda3\envs\yolov8\lib\site-packages
Requires: filelock, fsspec, jinja2, mkl, networkx, sympy, typing-extensions
Required-by: accelerate, thop, torchaudio, torchvision, ultralytics, ultralytics-thop
---
Name: accelerate
Version: 1.0.1
Summary: Accelerate
Home-page: https://github.com/huggingface/accelerate
Author: The HuggingFace team
Author-email: zach.mueller@huggingface.co
License: Apache
Location: d:\anaconda3\envs\yolov8\lib\site-packages
Requires: huggingface-hub, numpy, packaging, psutil, pyyaml, safetensors, torch
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [6]:
#使用modelscope下载模型
!pip install modelscope

Looking in indexes: https://mirrors.ustc.edu.cn/pypi/web/simple


In [ ]:
import gc
import torch
from modelscope import AutoModelForCausalLM, AutoTokenizer

### GPU用量检测和缓存清理

In [13]:
def print_gpu_memory(tag=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        free = total - allocated
        print(f"[{tag}] 已用: {allocated:.2f}GB | 缓存: {reserved:.2f}GB | 总计: {total:.2f}GB | 剩余: {free:.2f}GB")
    else:
        print("CUDA不可用")

In [ ]:
def cleanMemory(model):
    del model
    torch.cuda.empty_cache()
    gc.collect()
    print_gpu_memory("缓存已清理")

## Qwen/Qwen3-0.6B纯文本模型加载

下载模型

In [10]:
!modelscope download --model Qwen/Qwen3-0.6B --local_dir ./Qwen/Qwen3-0.6B

Traceback (most recent call last):
  File "D:\anaconda3\envs\yolov8\lib\runpy.py", line 194, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "D:\anaconda3\envs\yolov8\lib\runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "D:\anaconda3\envs\yolov8\Scripts\modelscope.exe\__main__.py", line 4, in <module>
  File "D:\anaconda3\envs\yolov8\lib\site-packages\modelscope\__init__.py", line 5, in <module>
    from modelscope.utils.import_utils import (LazyImportModule,
  File "D:\anaconda3\envs\yolov8\lib\site-packages\modelscope\utils\import_utils.py", line 18, in <module>
    from modelscope.utils.ast_utils import (INDEX_KEY, MODULE_KEY, REQUIREMENT_KEY,
  File "D:\anaconda3\envs\yolov8\lib\site-packages\modelscope\utils\ast_utils.py", line 24, in <module>
    from modelscope.utils.registry import default_group
  File "D:\anaconda3\envs\yolov8\lib\site-packages\modelscope\utils\registry.py", line 11, in <module>
    logger = get_logger()
  File "

加载Qwen/Qwen3-0.6B

In [ ]:
def load_model(model_name):
    # 加载分词器
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    # 加载模型
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.bfloat16,
        # dtype=torch.float16, #上面的不能用就用下面的
        device_map="auto"
    )
    return model, tokenizer

准备prompt并推理得到输出

In [ ]:
# 模型存放地址或模型名
model_name = "./Qwen/Qwen3-0.6B"
# 准备 prompt
prompt = "Give me a short introduction to large language model."
# 封装到 messages
messages = [
    {"role": "user", "content": prompt}
]
# 加载模型
print_gpu_memory("加载模型前")
model, tokenizer = load_model(model_name)
print_gpu_memory("加载模型后")

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True # 是否使用思考模式
)
# 编码
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
# 推理
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768 #最大上下文长度
)
# 得到输出并转 ids
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
try:
    # 找到思考结束的符号的id的位置 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0
# 思考内容
thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
# 输出内容
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)

In [ ]:
# 清理缓存
CleanMemory(model)

## Qwen/Qwen3-0.6B纯文本模型本地部署

安装vllm和openai库

In [ ]:
# !pip install vllm openai
pip show vllm openai

部署在本地的8000端口

In [ ]:
!python -m vllm.entrypoints.openai.api_server --model ./Qwen/Qwen3-0.6B --host 127.0.0.1 --port 8000

使用OpenAI标准接口调用

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:8000/v1",  # 服务提供商的地址，本地qwen系列这样写
    api_key="anything"                    # 本地模型的apikey可以随便填
)

response = client.chat.completions.create(
    model="./Qwen/Qwen3-0.6B",
    messages=[
        {"role": "user", "content": "介绍一下你自己 /no_think"}  # /no_think表示不思考， /think表示要思考
    ],
    max_tokens=32768,
    stream=False  # 流式对话，逐字返回而不是等全部生成完再返回
)

print(response.choices[0].message.content)

使用模型进行多轮会话

方案一：最简单的写法

In [ ]:
from openai import OpenAI

client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="123456")

# messages = [{"role": "system", "content": "你是一个有帮助的助手。"}]  #系统提示词
messages = []

while True:
    user_input = input("我: ")
    user_input = user_input + " /no_think"  # 默认思考，加/no_think表示不思考
    if user_input.lower() == 'q': #输入q退出
        break
    
    messages.append({"role": "user", "content": user_input})
    
    response = client.chat.completions.create(
        model="./Qwen/Qwen3-0.6B",
        messages=messages,
        max_tokens=32768
    )
    
    reply = response.choices[0].message.content
    print(f"Qwen3: {reply}\n")
    
    messages.append({"role": "assistant", "content": reply})

方案二：流式输出

In [ ]:
from openai import OpenAI
from IPython.display import display, Markdown, clear_output

client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="abcdefg")

# messages = [{"role": "system", "content": "你是一个有帮助的助手。"}]
message = []

while True:
    user_input = input("我: ")    # 默认思考
    if user_input.lower() == 'q': # 输入q退出
        break
    
    messages.append({"role": "user", "content": user_input})
    
    # 流式请求
    stream = client.chat.completions.create(
        model="./Qwen/Qwen3-0.6B",
        messages=messages,
        stream=True,
        max_tokens=32768
    )
    
    # Jupyter 流式显示：不断刷新同一个输出区域
    full_response = ""
    for chunk in stream:
        delta = chunk.choices[0].delta
        if delta.content is not None:
            full_response += delta.content
            clear_output(wait=True)          # 清除上一次显示
            display(Markdown(f"Qwen3: {full_response}"))  # 重新渲染完整内容
    
    print()  # 换行
    messages.append({"role": "assistant", "content": full_response})

## 模型量化

模型量化就是通过降低模型参数的数值精度（如从16位浮点数降到4位整数），以极小的性能损失换取模型体积缩小、显存占用降低和推理速度提升的压缩技术。

动态量化：不需要提前下载量化模型，框架会在加载时自动把 FP16/BF16 的模型压缩成 8-bit或4-bit，极大节省显存。

In [18]:
# 检查版本是新的
!pip install -U bitsandbytes

Looking in indexes: https://mirrors.ustc.edu.cn/pypi/web/simple


8-bit量化

In [ ]:
def load_model_8bit(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    # 8bit量化配置
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        #load_in_8bit=True,                  # 直接加这一句就行，已经过时
        quantization_config=bnb_config,      # 新的写法
        device_map="auto"
    )
    return model, tokenizer

model_name = "./Qwen/Qwen3-0.6B"
print_gpu_memory("加载8bit量化模型前")
load_model_8bit(model_name)
print_gpu_memory("加载8bit量化模型后")

4-bit量化
4-bit压缩非常狠，如果用简单的压缩方法，模型会直接“变傻”，BitsAndBytes 库使用了很多复杂的补救算法。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

def load_model_4bit(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    # 1. 定义量化配置
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,                       # 开启 4-bit 量化
        bnb_4bit_compute_dtype=torch.bfloat16,   # 计算时使用的数据类型，Tensor Core不支持 4-bit的数学运算，计算时还原成fp16
        #bnb_4bit_compute_dtype=torch.float16,   # 上面不能用就用下面的
        bnb_4bit_quant_type="nf4",               # 量化类型，nf4 效果最好
        bnb_4bit_use_double_quant=True,          # 使用双量化，再省一点点显存，对缩放因子再做一次量化
    )

    # 2. 加载模型时传入 quantization_config
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,  # 传入量化配置
        device_map="auto"
    )
    return model, tokenizer

model_name = "./Qwen/Qwen3-0.6B"
print_gpu_memory("加载4bit量化模型前")
load_model_4bit(model_name)
print_gpu_memory("加载4bit量化模型后")

### 加载微调所需的数据

In [ ]:
import os
import json
import random
import time

# 国内用户可设置 HF_ENDPOINT=https://hf-mirror.com 使用镜像加速
# 默认使用 HuggingFace 官方源
if "HF_ENDPOINT" not in os.environ:
    os.environ["HF_ENDPOINT"] = "https://huggingface.co"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

from huggingface_hub import hf_hub_download

endpoint = os.environ["HF_ENDPOINT"]
print(f"下载 shibing624/medical 数据集...")
print(f"源: {endpoint}")

repo_id = "shibing624/medical"

files_to_download = [
    ("finetune/train_zh_0.json", "训练集"),
    ("finetune/valid_zh_0.json", "验证集"),
    ("finetune/test_zh_0.json", "测试集"),
]

downloaded_files = []
for filename, desc in files_to_download:
    print(f"\n下载 {desc}: {filename}...")
    for attempt in range(3):
        try:
            fpath = hf_hub_download(
                repo_id=repo_id,
                filename=filename,
                repo_type="dataset",
                local_dir="data/medical_raw",
                force_download=True,
            )
            print(f"  下载成功: {fpath}")
            downloaded_files.append((fpath, desc))
            break
        except Exception as e:
            print(f"  第 {attempt+1} 次尝试失败: {e}")
            if attempt < 2:
                print(f"  等待 5 秒后重试...")
                time.sleep(5)
            else:
                print(f"  {desc} 下载失败，跳过")

if not downloaded_files:
    print("\n所有文件下载失败！")
    exit(1)

print("\n读取并处理数据...")
all_data = []
for fpath, desc in downloaded_files:
    print(f"读取 {desc}: {fpath}...")
    with open(fpath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    item = json.loads(line)
                    all_data.append(item)
                except json.JSONDecodeError:
                    pass

print(f"共读取 {len(all_data)} 条数据")

if all_data:
    print(f"数据字段: {list(all_data[0].keys())}")
    print(f"\n前3条数据示例:")
    for i in range(min(3, len(all_data))):
        print(f"\n--- 示例 {i+1} ---")
        print(json.dumps(all_data[i], ensure_ascii=False, indent=2)[:500])

    sample_size = min(10000, len(all_data))
    random.seed(42)
    random.shuffle(all_data)
    samples = all_data[:sample_size]

    train_size = int(sample_size * 0.9)
    valid_size = sample_size - train_size

    train_data = samples[:train_size]
    valid_data = samples[train_size:]

    os.makedirs("data/medical", exist_ok=True)

    train_file = "data/medical/train.jsonl"
    with open(train_file, "w", encoding="utf-8") as f:
        for item in train_data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    valid_file = "data/medical/valid.jsonl"
    with open(valid_file, "w", encoding="utf-8") as f:
        for item in valid_data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print(f"\n已保存 {train_size} 条训练数据到 {train_file}")
    print(f"已保存 {valid_size} 条验证数据到 {valid_file}")
    print("\n数据集下载和处理完成！")
else:
    print("未读取到数据。")